In [ ]:
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt
from sklearn.neighbors import KernelDensity
import numpy as np
import pandas as pd

# Load the geojson data
gdf = gpd.read_file('311_Encampment_Reports%2C_City_of_San_Diego%2C_2022.geojson')


In [ ]:

# # Extract geometry (points) and convert them to numpy array for kernel density estimation
# coords = np.array([[point.x, point.y] for point in gdf.geometry])

# # Perform Kernel Density Estimation (KDE)
# kde = KernelDensity(bandwidth=0.005, kernel='gaussian')
# kde.fit(coords)

# # Create a grid to evaluate the KDE
# x_min, y_min = coords.min(axis=0)
# x_max, y_max = coords.max(axis=0)

# x_grid = np.linspace(x_min, x_max, 500)
# y_grid = np.linspace(y_min, y_max, 500)
# x_mesh, y_mesh = np.meshgrid(x_grid, y_grid)
# xy_sample = np.vstack([x_mesh.ravel(), y_mesh.ravel()]).T

# # Evaluate the KDE on this grid
# z = np.exp(kde.score_samples(xy_sample)).reshape(x_mesh.shape)

# # 1. Plot with only point data overlaid on the map
# fig, ax = plt.subplots(figsize=(10, 10))

# # Plot the encampment locations
# gdf.plot(ax=ax, marker='o', color='red', markersize=2, label="Encampment Locations")

# # Add the basemap
# ctx.add_basemap(ax, crs=gdf.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.6)

# plt.title("Point Data of Encampment Reports (2022) Overlaid on San Diego County Map")
# plt.xlabel("Longitude")
# plt.ylabel("Latitude")
# plt.legend()
# plt.show()


In [ ]:

# # 2. Plot with only the kernel density heatmap overlaid on the map
# fig, ax = plt.subplots(figsize=(10, 10))

# # Plot the kernel density estimation map
# ax.imshow(z, origin='lower', cmap='inferno', extent=[x_min, x_max, y_min, y_max], alpha=1)

# # Add the basemap
# ctx.add_basemap(ax, crs=gdf.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.4)

# plt.title("Kernel Density Heatmap of Encampment Reports (2022) Overlaid on San Diego County Map")
# plt.xlabel("Longitude")
# plt.ylabel("Latitude")
# plt.show()


In [ ]:
# # Define the coordinates for zooming in (specific region)
# # Set custom longitude and latitude limits (you can adjust these based on your desired zoom area)
# zoom_x_min = -117.25  # Example longitude min
# zoom_x_max = -117.15  # Example longitude max
# zoom_y_min = 32.70    # Example latitude min
# zoom_y_max = 32.80    # Example latitude max

# # Plot the kernel density estimation map with a zoomed-in view
# fig, ax = plt.subplots(figsize=(10, 10))

# # Adjust the extent to zoom into a specific area
# ax.imshow(z, origin='lower', cmap='inferno', extent=[zoom_x_min, zoom_x_max, zoom_y_min, zoom_y_max], alpha=1)

# # Add the basemap (zooming in as well)
# ctx.add_basemap(ax, crs=gdf.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.4, zoom=14)

# plt.title("Zoomed-in Kernel Density Heatmap")
# plt.xlabel("Longitude")
# plt.ylabel("Latitude")
# plt.show()

## Okay now Kaelan. CMON use your fkcn brain here. You need to break this down by different time periods. and also asjust different parameters of the kernel density calculations

In [ ]:
gdf.head()

In [ ]:
gdf.shape

In [ ]:
gdf.columns

In [ ]:
gdf['date_reque'].dtype

In [ ]:
gdf['date_reque'] = pd.to_datetime(gdf['date_reque'])


In [ ]:
gdf['year_month'] = gdf['date_reque'].dt.to_period('M')
gdf['year_week'] = gdf['date_reque'].dt.to_period('W')
gdf['year_day'] = gdf['date_reque'].dt.to_period('D')


In [ ]:
gdf.head()

In [ ]:
# Redefine the seasons based on months
spring_months = [3, 4, 5]
summer_months = [6, 7, 8]
fall_months = [9, 10, 11]
winter_months = [12, 1, 2]

# Extract the month from the 'date_reque' column
gdf['month'] = gdf['date_reque'].dt.month

# Create separate dataframes for each season
spring_df = gdf[gdf['month'].isin(spring_months)]
summer_df = gdf[gdf['month'].isin(summer_months)]
fall_df = gdf[gdf['month'].isin(fall_months)]
winter_df = gdf[gdf['month'].isin(winter_months)]

In [ ]:
summer_df.shape

In [ ]:
# Function to assign seasons based on the month
def assign_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Fall'

# Apply the function to the 'month' column
gdf['season'] = gdf['month'].apply(assign_season)

# Plot the distribution of data points by season
season_distribution = gdf['season'].value_counts().reindex(['Winter', 'Spring', 'Summer', 'Fall'])

fig, ax = plt.subplots(figsize=(10, 6))
season_distribution.plot(kind='bar', color='lightcoral', ax=ax)
ax.set_title('Distribution of Data Points by Season 2022 (Winter, Spring, Summer, Fall)')
ax.set_xlabel('Season')
ax.set_ylabel('Number of Data Points')
plt.show()


In [ ]:
# Create individual dataframes for each month
jan_df = gdf[gdf['month'] == 1]
feb_df = gdf[gdf['month'] == 2]
mar_df = gdf[gdf['month'] == 3]
apr_df = gdf[gdf['month'] == 4]
may_df = gdf[gdf['month'] == 5]
jun_df = gdf[gdf['month'] == 6]
jul_df = gdf[gdf['month'] == 7]
aug_df = gdf[gdf['month'] == 8]
sep_df = gdf[gdf['month'] == 9]
oct_df = gdf[gdf['month'] == 10]
nov_df = gdf[gdf['month'] == 11]
dec_df = gdf[gdf['month'] == 12]

# Confirm that the dataframes were created
(jan_df.shape, feb_df.shape, mar_df.shape, apr_df.shape, may_df.shape, jun_df.shape,
 jul_df.shape, aug_df.shape, sep_df.shape, oct_df.shape, nov_df.shape, dec_df.shape)


In [ ]:
# Create a dictionary of all months with their respective number of data points
all_month_data = {
    'January': jan_df.shape[0],
    'February': feb_df.shape[0],
    'March': mar_df.shape[0],
    'April': apr_df.shape[0],
    'May': may_df.shape[0],
    'June': jun_df.shape[0],
    'July': jul_df.shape[0],
    'August': aug_df.shape[0],
    'September': sep_df.shape[0],
    'October': oct_df.shape[0],
    'November': nov_df.shape[0],
    'December': dec_df.shape[0]
}

# Create a bar plot to show the distribution of points across all months
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(all_month_data.keys(), all_month_data.values(), color='cornflowerblue')
ax.set_title('Distribution of Data Points Across All Months (2022)')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Data Points')
ax.set_xticklabels(all_month_data.keys(), rotation=45)
plt.show()


In [ ]:
# Calculate the global extent (bounding box) for all 12 months
x_min_global = gdf.geometry.x.min()
x_max_global = gdf.geometry.x.max()
y_min_global = gdf.geometry.y.min()
y_max_global = gdf.geometry.y.max()

In [ ]:
def plot_kernel_density(gdf, title, bandwidth=0.005):
    # Ensure geometries are valid
    gdf = gdf[gdf.geometry.notnull()]
    
    # Extract geometry points
    coords = np.array([[point.x, point.y] for point in gdf.geometry])
    
    # Perform Kernel Density Estimation (KDE)
    kde = KernelDensity(bandwidth=bandwidth, kernel='gaussian')
    kde.fit(coords)
    
    # # Create a grid to evaluate the KDE
    # x_min, y_min = coords.min(axis=0)
    # x_max, y_max = coords.max(axis=0)

    x_grid = np.linspace(x_min_global, x_max_global, 500)
    y_grid = np.linspace(y_min_global, y_max_global, 500)
    x_mesh, y_mesh = np.meshgrid(x_grid, y_grid)
    xy_sample = np.vstack([x_mesh.ravel(), y_mesh.ravel()]).T

    # Evaluate the KDE on this grid
    z = np.exp(kde.score_samples(xy_sample)).reshape(x_mesh.shape)

    # Plot the KDE with a basemap
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(z, origin='lower', cmap='inferno', extent=[x_min_global, x_max_global, y_min_global, y_max_global], alpha=1)
    
    # Add basemap
    ctx.add_basemap(ax, crs=gdf.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.4)
    
    plt.title(title)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.show()

# Generate kernel density maps for Winter 2020, March 2020, and Week 10 of 2020
# plot_kernel_density(winter_df, "Winter 2020 Kernel Density Map")
# plot_kernel_density(mar_df, "March 2020 Kernel Density Map")
#plot_kernel_density(week10_df, "Week 10, 2020 Kernel Density Map")


In [ ]:
plot_kernel_density(gdf, "2022")

In [ ]:
# Loop to generate monthly kernel density maps for 2022
for month in range(1, 13):
    month_df = gdf[(gdf['month'] == month) & (gdf['date_reque'].dt.year == 2022)]
    title = f"2022/{month:02d}"
    plot_kernel_density(month_df, title)

In [ ]:
# Loop to generate seasonal kernel density maps for 2020
for season in ['Winter', 'Spring', 'Summer', 'Fall']:
    # Filter the data for the current season and year 2020
    season_df = gdf[(gdf['season'] == season) & (gdf['date_reque'].dt.year == 2022)]
    
    # Create the title for the plot
    title = f"2022/{season}"
    
    # Plot the kernel density map for this season
    plot_kernel_density(season_df, title)

In [ ]:
# Ensure the 'date_requested' column is properly converted to datetime
gdf['date_reque'] = pd.to_datetime(gdf['date_reque'])

# Create columns for month and week
gdf['month'] = gdf['date_reque'].dt.month
gdf['week'] = gdf['date_reque'].dt.isocalendar().week  # Extract week number from the date

# Filter data for August 2022 (Month = 8)
aug_df = gdf[(gdf['month'] == 8) & (gdf['date_reque'].dt.year == 2022)]

# Find the unique weeks in July
unique_weeks_in_aug = aug_df['week'].unique()

In [ ]:
# Loop to generate weekly kernel density maps for each week in August 2022
for week in unique_weeks_in_aug:
    # Filter data for the specific week in August 2022
    week_df = aug_df[aug_df['week'] == week]
    
    # Create the title for the plot
    title = f"2022/Week{week}"
    
    # Plot the kernel density map for this week
    plot_kernel_density(week_df, title)